# DeepSeek V3.2 模型结构完整实现

## 一、完整模型代码（ModelArgs / 各核心模块 / Transformer）

In [ ]:
import math  # 导入数学库，用于log/sqrt等运算（RoPE频率、注意力缩放等）
from dataclasses import dataclass  # 导入dataclass装饰器，用于简洁地定义模型超参数数据类
from typing import Tuple, Optional, Literal  # 导入类型注解：元组、可选值、字面量类型

import torch  # 导入PyTorch核心库
from torch import nn  # 导入神经网络模块，提供Module/Parameter等基础组件
import torch.nn.functional as F  # 导入函数式API，提供softmax/embedding/layer_norm等无状态函数
import torch.distributed as dist  # 导入分布式通信模块，用于多GPU张量并行

from kernel import act_quant, fp8_gemm, fp8_index  # 从本地kernel模块导入自定义FP8量化和矩阵乘函数


# 全局分布式状态，默认单机单卡
world_size = 1   # 参与分布式推理的总进程数（GPU数量），默认1
rank = 0         # 当前进程的全局编号，默认0（主进程）
block_size = 128  # FP8量化的分组大小：每128个元素共享一个缩放因子

@dataclass
class ModelArgs:
    """
    Data class for defining model arguments and hyperparameters.

    Attributes:
        max_batch_size (int): Maximum batch size.
        max_seq_len (int): Maximum sequence length.
        dtype (Literal["bf16", "fp8"]): Data type for computations.
        scale_fmt (Optional[str]): Format for quantization scale.
        vocab_size (int): Vocabulary size.
        dim (int): Model dimension.
        inter_dim (int): Intermediate dimension for MLP layers.
        moe_inter_dim (int): Intermediate dimension for MoE layers.
        n_layers (int): Number of transformer layers.
        n_dense_layers (int): Number of dense layers in the model.
        n_heads (int): Number of attention heads.
        n_routed_experts (int): Number of routed experts for MoE layers.
        n_shared_experts (int): Number of shared experts for MoE layers.
        n_activated_experts (int): Number of activated experts in MoE layers.
        n_expert_groups (int): Number of expert groups.
        n_limited_groups (int): Number of limited groups for MoE routing.
        score_func (Literal["softmax", "sigmoid"]): Scoring function for MoE routing.
        route_scale (float): Scaling factor for routing scores.
        q_lora_rank (int): LoRA rank for query projections.
        kv_lora_rank (int): LoRA rank for key-value projections.
        qk_nope_head_dim (int): Dimension for query-key projections without positional embeddings.
        qk_rope_head_dim (int): Dimension for query-key projections with rotary embeddings.
        v_head_dim (int): Dimension for value projections.
        original_seq_len (int): Original sequence length.
        rope_theta (float): Base for rotary positional encoding.
        rope_factor (float): Scaling factor for extended sequence lengths.
        beta_fast (int): Fast beta correction factor.
        beta_slow (int): Slow beta correction factor.
        mscale (float): Scaling factor for extended attention.
        index_head_dim (int): Dimension for index head.
        index_topk (int): Top-k for index head.
    """
    max_batch_size: int = 8             # 最大批量大小，决定KV缓存预分配的第一维
    max_seq_len: int = 4096 * 4         # 最大序列长度（16384），支持长上下文
    dtype: Literal["bf16", "fp8"] = "bf16"  # 权重精度：bf16或fp8量化
    scale_fmt: Optional[str] = None     # FP8缩放因子格式，None表示连续缩放
    vocab_size: int = 102400            # 词汇表大小
    dim: int = 2048                     # 模型隐藏层维度（d_model）
    inter_dim: int = 10944              # 密集FFN中间层维度
    moe_inter_dim: int = 1408           # MoE专家FFN中间层维度（比inter_dim小）
    n_layers: int = 27                  # Transformer总层数
    n_dense_layers: int = 1             # 使用密集MLP的前n层数量
    n_heads: int = 16                   # 注意力头数
    # --- MoE 相关参数 ---
    n_routed_experts: int = 16          # 路由专家总数（此处改为16以降低显存需求；原版为64）
    n_shared_experts: int = 2           # 共享专家数（每次都激活）
    n_activated_experts: int = 6        # 每token激活的路由专家数
    n_expert_groups: int = 1            # 专家分组数，1表示不分组
    n_limited_groups: int = 1           # 每次激活的最大专家组数
    score_func: Literal["softmax", "sigmoid"] = "softmax"  # MoE路由得分函数
    route_scale: float = 1.             # 路由权重缩放系数
    # --- MLA 相关参数 ---
    q_lora_rank: int = 0                # Query低秩秩，0表示不使用低秩分解
    kv_lora_rank: int = 512             # KV低秩秩（MLA核心压缩参数）
    qk_nope_head_dim: int = 128         # QK非位置编码部分每头维度
    qk_rope_head_dim: int = 64          # QK位置编码（RoPE）部分每头维度
    v_head_dim: int = 128               # Value每头维度
    # --- YaRN 位置外推参数 ---
    original_seq_len: int = 4096        # 预训练时的原始最大序列长度
    rope_theta: float = 10000.0         # RoPE基础频率theta
    rope_factor: float = 40             # YaRN长度外推缩放因子
    beta_fast: int = 32                 # YaRN快速插值beta参数（高频边界）
    beta_slow: int = 1                  # YaRN慢速插值beta参数（低频边界）
    mscale: float = 1.                  # YaRN注意力缩放修正系数
    # --- Indexer 相关参数 ---
    index_n_heads: int = 64             # Indexer模块的注意力头数
    index_head_dim: int = 128           # Indexer每个头的维度
    index_topk: int = 2048              # Indexer每次选择的top-k候选key数量

class ParallelEmbedding(nn.Module):
    """
    Embedding layer with parallelism support across distributed processes.

    Args:
        vocab_size (int): Vocabulary size.
        dim (int): Embedding dimension.
    """
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()  # 调用父类nn.Module初始化
        self.vocab_size = vocab_size  # 保存词汇表总大小
        self.dim = dim  # 保存嵌入向量维度
        assert vocab_size % world_size == 0, f"Vocabulary size must be divisible by world size (world_size={world_size})"  # 断言词汇表大小能被进程数整除
        self.part_vocab_size = (vocab_size // world_size)  # 当前进程负责的词汇表分片大小
        self.vocab_start_idx = rank * self.part_vocab_size  # 当前进程的词汇表起始索引（闭区间）
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size  # 当前进程的词汇表结束索引（开区间）
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))  # 本地嵌入矩阵参数，形状(part_vocab_size, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for parallel embedding layer.

        Args:
            x (torch.Tensor): Input tensor containing token indices.

        Returns:
            torch.Tensor: Embedded representations.

        Raises:
            ValueError: If `world_size` is not defined.
        """
        if world_size > 1:  # 多GPU模式：只查找本进程负责的词汇表范围
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)  # 标记不属于本进程词汇表范围的token，True=不属于本进程
            x = x - self.vocab_start_idx  # 将token id转为本进程的局部偏移量
            x[mask] = 0  # 将越界token id置0（查找后会得到0向量，后续会清零）
        y = F.embedding(x, self.weight)  # 查找本地嵌入矩阵，形状(batch, seq, dim)
        if world_size > 1:  # 多GPU模式：聚合各进程的查找结果
            y[mask] = 0  # 将不属于本进程的位置清零（防止引入随机初始化值）
            dist.all_reduce(y)  # all_reduce求和，每个位置只有一个进程有非零值
        return y  # 返回嵌入向量，形状(batch, seq, dim)


def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None,
           scale_fmt: Optional[str] = None) -> torch.Tensor:
    """
    Applies a linear transformation to the incoming data: y = xA^T + b.
    This function supports specialized implementations based on quantization
    and tensor formats.

    Args:
        x (torch.Tensor): The input tensor.
        weight (torch.Tensor): The weight tensor. It may be quantized and
            requires dequantization for certain cases.
        bias (Optional[torch.Tensor]): The bias tensor to be added. Default is None.
        scale_fmt (Optional[str]): The format of scaling factors.

    Returns:
        torch.Tensor: The result of the linear transformation, which may involve
        quantization-aware computations depending on the input parameters.

    Notes:
        - If `weight` is quantized (e.g., `element_size() == 1`), a dequantized version
          is used for computation.
        - For other cases, the function applies quantization to `x` and uses `fp8_gemm` for computation.
    """
    assert bias is None  # 当前版本不支持偏置（FP8 GEMM接口暂不支持bias融合）

    if weight.dtype != torch.float8_e4m3fn:  # 非FP8权重（BF16或FP32）
        return F.linear(x, weight)  # 直接使用PyTorch标准线性变换：y = x @ weight^T
    else:  # FP8量化权重路径
        x, scale = act_quant(x, block_size, scale_fmt)  # 对输入x进行FP8量化，返回量化张量和缩放因子
        return fp8_gemm(x, scale, weight, weight.scale)  # 使用自定义FP8 GEMM执行量化矩阵乘法


class Linear(nn.Module):
    """
    Custom linear layer with support for quantized weights and optional bias.

    Args:
        in_features (int): Number of input features.
        out_features (int): Number of output features.
        bias (bool): Whether to include a bias term. Defaults to False.
        dtype (optional): Data type for the layer. Defaults to `torch.bfloat16`.
    """
    dtype = torch.bfloat16      # 类变量：全局默认权重类型，由Transformer.__init__设置
    scale_fmt: Optional[str] = None  # 类变量：FP8缩放因子格式，None表示连续缩放

    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        super().__init__()  # 调用父类nn.Module初始化
        self.in_features = in_features    # 保存输入特征维度
        self.out_features = out_features  # 保存输出特征维度
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype or Linear.dtype))  # 初始化权重参数，形状(out, in)；dtype未指定时使用类变量
        if self.weight.element_size() == 1:  # 如果权重是FP8（element_size=1字节）
            scale_out_features = (out_features + block_size - 1) // block_size  # 输出维度方向的缩放因子数量（向上取整）
            scale_in_features = (in_features + block_size - 1) // block_size    # 输入维度方向的缩放因子数量（向上取整）
            self.weight.scale = self.scale = nn.Parameter(torch.empty(scale_out_features, scale_in_features, dtype=torch.float32))  # 为FP8权重创建block-wise缩放因子参数，形状(out//128, in//128)
        else:  # 非FP8权重不需要缩放因子
            self.register_parameter("scale", None)  # 注册为None参数（保持接口一致）
        if bias:  # 如果需要偏置
            self.bias = nn.Parameter(torch.empty(out_features))  # 初始化偏置，形状(out_features,)
        else:  # 不需要偏置
            self.register_parameter("bias", None)  # 注册为None参数

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the custom linear layer.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Transformed tensor after linear computation.
        """
        return linear(x, self.weight, self.bias, self.scale_fmt)  # 调用通用linear函数执行实际计算


class ColumnParallelLinear(Linear):
    """
    列并行线性层：将输出特征维度（out_features）按进程数均匀切分。

    每个进程只持有 out_features // world_size 个输出神经元对应的权重。
    常用于MLP的w1/w3（升维）、MHA的QKV投影等。

    Args:
        in_features (int): Number of input features.
        out_features (int): Total number of output features.
        bias (bool): Whether to include a bias term. Defaults to False.
        dtype (optional): Data type for the layer. Defaults to `torch.bfloat16`.
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        assert out_features % world_size == 0, f"Output features must be divisible by world size (world_size={world_size})"  # 断言输出维度可被进程数整除
        self.part_out_features = out_features // world_size  # 计算本进程负责的输出特征数
        super().__init__(in_features, self.part_out_features, bias, dtype)  # 只创建本进程部分的权重

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for column parallel linear layer.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Transformed tensor with column-parallel computation.
        """
        y = linear(x, self.weight, self.bias, self.scale_fmt)  # 计算本进程负责的列部分输出，形状(..., part_out_features)
        return y  # 返回本地输出（完整输出需all_gather）


class RowParallelLinear(Linear):
    """
    行并行线性层：将输入特征维度（in_features）按进程数均匀切分。

    每个进程只持有 in_features // world_size 个输入维度对应的权重行。
    通过all_reduce求和得到完整输出，常用于MLP的w2（降维）、注意力输出投影等。

    Args:
        in_features (int): Total number of input features.
        out_features (int): Number of output features.
        bias (bool): Whether to include a bias term. Defaults to False.
        dtype (optional): Data type for the layer. Defaults to `torch.bfloat16`.
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False,
                 reduce_output=True, dtype=None):
        assert in_features % world_size == 0, f"Input features must be divisible by world size (world_size={world_size})"  # 断言输入维度可被进程数整除
        self.part_in_features = in_features // world_size  # 计算本进程负责的输入特征数
        self.reduce_output = reduce_output  # 保存是否需要all_reduce的标志
        super().__init__(self.part_in_features, out_features, bias, dtype)  # 只创建本进程部分的权重

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for row parallel linear layer.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Transformed tensor with row-parallel computation.
        """
        y = linear(x, self.weight, None, self.scale_fmt)  # 计算本进程的部分内积（行并行分片）
        if self.reduce_output and world_size > 1:  # 需要聚合且是多GPU模式
            y = y.float()  # 转float32提高all_reduce精度
            dist.all_reduce(y)  # 对所有进程的部分内积求和，得到完整输出
        if self.bias is not None:  # 如果有偏置
            y += self.bias  # 加上偏置向量
        return y.type_as(x)  # 恢复为输入dtype并返回


class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization (RMSNorm).

    Args:
        dim (int): Dimension of the input tensor.
        eps (float): Epsilon value for numerical stability. Defaults to 1e-6.
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()  # 调用父类初始化
        self.dim = dim    # 保存特征维度
        self.eps = eps    # 保存数值稳定项，防止除零
        self.weight = nn.Parameter(torch.ones(dim, dtype=torch.float32))  # 可学习缩放参数，全1初始化，使用float32提高归一化精度

    def forward(self, x: torch.Tensor, residual: Optional[torch.Tensor] = None):
        """
        RMSNorm前向传播，支持可选的残差融合以减少内存读写。

        Args:
            x (torch.Tensor): Input tensor, shape (..., dim)
            residual (Optional[torch.Tensor]): 可选残差张量，若提供则先执行 x = x + residual

        Returns:
            当residual为None时：归一化结果，形状(..., dim)
            当residual不为None时：(归一化结果, 更新后的residual)
        """
        dtype = x.dtype  # 记录输入原始dtype，用于最后恢复类型
        if residual is None:  # 无残差模式：直接归一化
            x = x.float()  # 转float32进行高精度计算（BF16精度不足以保证归一化稳定）
            var = x.pow(2).mean(-1, keepdim=True)  # 计算最后一维的均方值（RMS的平方），形状(..., 1)
            x = x * torch.rsqrt(var + self.eps)  # 乘以RMS倒数完成归一化：x / sqrt(mean(x^2) + eps)
            return (self.weight * x).to(dtype)  # 乘以可学习缩放参数后恢复原始dtype
        else:  # 有残差模式：先融合残差再归一化
            x = residual = x.float() + residual.float()  # 计算残差融合：新x = 旧x + residual（均用float32），同时更新residual
            var = x.pow(2).mean(-1, keepdim=True)  # 计算融合后张量的均方值，形状(..., 1)
            x = x * torch.rsqrt(var + self.eps)  # 归一化操作
            return (self.weight * x).to(dtype), residual.to(dtype)  # 返回归一化结果和更新后的残差（均恢复原dtype）


class LayerNorm(nn.Module):
    """
    标准层归一化（Layer Normalization），用于Indexer模块的Key归一化。
    公式：LayerNorm(x) = (x - mean(x)) / std(x) * weight + bias
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()  # 调用父类初始化
        self.dim = dim  # 特征维度
        self.eps = eps  # 数值稳定项
        self.weight = nn.Parameter(torch.ones(dim, dtype=torch.float32))   # 可学习缩放参数，全1初始化
        self.bias = nn.Parameter(torch.zeros(dim, dtype=torch.float32))    # 可学习偏移参数，全0初始化

    def forward(self, x: torch.Tensor):
        return F.layer_norm(x.float(), (self.dim,), self.weight, self.bias, self.eps).type_as(x)
        # 先转float32进行高精度LayerNorm，结果再转回输入dtype


def precompute_freqs_cis(args: ModelArgs) -> torch.Tensor:
    """
    Precomputes frequency-based complex exponential values for rotary positional embeddings.

    Args:
        args (ModelArgs): Model arguments containing positional embedding parameters.

    Returns:
        torch.Tensor: Precomputed complex exponential values for positional embeddings.
    """
    dim = args.qk_rope_head_dim
    seqlen = args.max_seq_len
    beta_fast = args.beta_fast
    beta_slow = args.beta_slow
    base = args.rope_theta
    factor = args.rope_factor

    def find_correction_dim(num_rotations, dim, base, max_seq_len):
        """
        Computes the correction dimension for a given number of rotations in the rotary positional embedding.

        Args:
            num_rotations (float): Number of rotations to compute the correction for.
            dim (int): Dimensionality of the embedding space.
            base (float): Base value for the exponential computation.
            max_seq_len (int): Maximum sequence length.

        Returns:
            float: The correction dimension based on the input parameters.
        """
        return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))

    def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
        """
        Computes the range of correction dimensions for rotary positional embeddings.

        Args:
            low_rot (float): Lower bound for the number of rotations.
            high_rot (float): Upper bound for the number of rotations.
            dim (int): Dimensionality of the embedding space.
            base (float): Base value for the exponential computation.
            max_seq_len (int): Maximum sequence length.

        Returns:
            Tuple[int, int]: The range of correction dimensions (low, high), clamped to valid indices.
        """
        low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))
        high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))
        return max(low, 0), min(high, dim-1)

    def linear_ramp_factor(min, max, dim):
        """
        Computes a linear ramp function used to smooth values between a minimum and maximum range.

        Args:
            min (float): Minimum value for the ramp function.
            max (float): Maximum value for the ramp function.
            dim (int): Dimensionality of the ramp tensor.

        Returns:
            torch.Tensor: A tensor of shape (dim,) with values linearly interpolated between 0 and 1,
                clamped to the range [0, 1].
        """
        if min == max:
            max += 0.001
        linear_func = (torch.arange(dim, dtype=torch.float32) - min) / (max - min)
        ramp_func = torch.clamp(linear_func, 0, 1)
        return ramp_func

    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
    # 计算标准RoPE基础频率：freqs[i] = 1 / theta^(2i/dim)，形状(dim//2,)
    if seqlen > args.original_seq_len:  # 如果序列长度超过预训练长度，启用YaRN外推
        low, high = find_correction_range(beta_fast, beta_slow, dim, base, args.original_seq_len)  # 计算YaRN插值的边界维度范围
        smooth = 1 - linear_ramp_factor(low, high, dim // 2)  # 计算平滑权重：高频smooth≈1，低频smooth≈0
        freqs = freqs / factor * (1 - smooth) + freqs * smooth  # YaRN混合：低频除以factor缩小，高频保持不变

    t = torch.arange(seqlen)  # 生成位置索引[0, 1, ..., seqlen-1]，形状(seqlen,)
    freqs = torch.outer(t, freqs)  # 外积：位置×频率，形状(seqlen, dim//2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # 转为复数：e^(j*freqs) = cos + j*sin，形状(seqlen, dim//2)
    return freqs_cis  # 返回预计算的RoPE复数因子


def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor, interleaved: bool = True) -> torch.Tensor:
    """
    Applies rotary positional embeddings to the input tensor.

    Args:
        x (torch.Tensor): Input tensor with positional embeddings to be applied.
        freqs_cis (torch.Tensor): Precomputed complex exponential values for positional embeddings.

    Returns:
        torch.Tensor: Tensor with rotary embeddings applied.
    """
    dtype = x.dtype   # 记录输入原始dtype
    shape = x.shape   # 记录输入形状
    if not interleaved:  # 非交错布局：需要先重排为交错格式
        x = x.view(*shape[:-1], 2, -1).transpose(-1, -2).contiguous()  # 将最后一维分成两半后转置
    x = torch.view_as_complex(x.float().view(*shape[:-1], -1, 2))
    # 将相邻两个float32转为一个复数：(..., rope_dim//2)的复数张量
    freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))  # 广播reshape：(1, seq, 1, rope_dim//2)
    y = torch.view_as_real(x * freqs_cis).flatten(3)  # 复数相乘（旋转）后转回实数，展平，形状(..., rope_dim)
    if not interleaved:  # 非交错布局：恢复原始维度顺序
        y = torch.cat([y[..., 0::2], y[..., 1::2]], dim=-1)  # 分离奇偶后拼接回非交错格式
    return y.to(dtype)  # 转回输入的原始dtype并返回


def rotate_activation(x: torch.Tensor) -> torch.Tensor:
    assert x.dtype == torch.bfloat16  # 断言输入为bfloat16（fast_hadamard_transform的要求）
    from fast_hadamard_transform import hadamard_transform  # 延迟导入，避免无GPU环境时导入失败
    hidden_size = x.size(-1)  # 获取最后一维大小（需为2的幂次）
    return hadamard_transform(x, scale=hidden_size ** -0.5)  # Hadamard变换并归一化（保持方差不变）


class Indexer(torch.nn.Module):
    """
    索引器模块 - 用于在长序列注意力计算中进行稀疏化，减少计算量。

    核心思想：
        不是计算 query 和所有 key 的注意力，而是先通过一个轻量级的索引机制
        快速筛选出最相关的 top-k 个位置，然后只在这些位置上计算完整的注意力。

    工作流程：
        1. 使用低秩投影将 query 和 key 映射到较小的索引空间
        2. 在索引空间中快速计算相似度得分（FP8 量化加速）
        3. 选出 top-k 个最相关的位置索引
        4. MLA 只在这些索引位置计算完整注意力（稀疏化）

    Args:
        args (ModelArgs): 模型超参数对象，需包含 dim、index_n_heads、
                          index_head_dim、qk_rope_head_dim、index_topk、
                          q_lora_rank、scale_fmt、max_batch_size、max_seq_len。
    """
    def __init__(self, args: ModelArgs):
        super().__init__()  # 调用父类初始化
        self.dim: int = args.dim              # 模型隐藏层维度
        self.n_heads: int = args.index_n_heads  # 索引器注意力头数
        self.n_local_heads = args.index_n_heads // world_size  # 本进程负责的头数
        self.head_dim: int = args.index_head_dim    # 索引器每个头的维度
        self.rope_head_dim: int = args.qk_rope_head_dim  # RoPE编码维度（与MLA共享）
        self.index_topk: int = args.index_topk      # 每次选择的top-k候选数
        self.q_lora_rank: int = args.q_lora_rank    # Query低秩秩
        self.wq_b = Linear(self.q_lora_rank, self.n_heads * self.head_dim)  # Query投影：低秩→多头索引空间
        self.wk = Linear(self.dim, self.head_dim)  # Key投影：隐藏层→单共享索引key（所有头共用）
        self.k_norm = LayerNorm(self.head_dim)  # Key的LayerNorm归一化，稳定训练
        # weights_proj存储为bf16，这里用fp32方便float计算时的精度
        self.weights_proj = Linear(self.dim, self.n_heads, dtype=torch.float32)  # 头权重投影：为每个头生成标量权重
        self.softmax_scale = self.head_dim ** -0.5  # 注意力缩放因子 1/sqrt(head_dim)
        self.scale_fmt = args.scale_fmt  # FP8缩放因子格式

        self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.head_dim, dtype=torch.float8_e4m3fn), persistent=False)
        # FP8格式的key缓存，形状(max_batch, max_seq, head_dim)，节省显存
        self.register_buffer("k_scale_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.head_dim // block_size, dtype=torch.float32), persistent=False)
        # key量化缩放因子缓存，形状(max_batch, max_seq, head_dim//128)


    def forward(self, x: torch.Tensor, qr: torch.Tensor, start_pos: int,
                freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        前向传播 - 计算索引得分并返回 top-k 位置索引。

        执行步骤：
            Step 1  从低秩 query 投影到多头索引 query 空间，并应用 RoPE
            Step 2  计算共享 key，做 LayerNorm 并应用 RoPE
            Step 3  对 query/key 应用 Hadamard 变换（增强特征分散性）
            Step 4  将 query/key 量化为 FP8 格式
            Step 5  将当前步 FP8 key 及缩放因子写入 KV 缓存
            Step 6  计算每个注意力头的标量权重并组合多个缩放因子
            Step 7  调用 FP8 核函数快速计算 Q @ K^T 索引得分
            Step 8  应用因果掩码（可选）
            Step 9  选出 top-k 个得分最高的 key 位置索引

        Args:
            x         (torch.Tensor): 输入特征，形状 (batch_size, seq_len, dim)
            qr        (torch.Tensor): 低秩 query 表示，形状 (batch_size, seq_len, q_lora_rank)
            start_pos (int)         : 当前序列在完整序列中的起始位置（用于 KV 缓存索引）
            freqs_cis (torch.Tensor): 预计算的 RoPE 复数因子，形状 (seq_len, rope_dim//2)
            mask      (Optional[torch.Tensor]): 因果注意力掩码，形状 (seq_len, seq_len)，
                                                None 表示 decode 单步推理

        Returns:
            topk_indices (torch.Tensor): 每个 query 位置选出的 top-k 个 key 位置索引，
                                         形状 (batch_size, seq_len, n_heads, index_topk)
        """
        bsz, seqlen, _ = x.size()  # 解包批量大小、序列长度、特征维度
        end_pos = start_pos + seqlen  # 计算当前序列在KV缓存中的结束位置
        q = self.wq_b(qr)  # 低秩query→多头索引query，形状(batch, seq, n_heads*head_dim)
        q = q.view(bsz, seqlen, self.n_heads, self.head_dim)  # 重塑为多头形式
        q_pe, q_nope = torch.split(q, [self.rope_head_dim, self.head_dim - self.rope_head_dim], dim=-1)
        # 分离query的RoPE部分和非RoPE部分
        # 注意：索引器使用非交错RoPE（interleaved=False），与MLA主注意力不同
        q_pe = apply_rotary_emb(q_pe, freqs_cis, False)  # 对RoPE部分应用旋转位置编码（非交错模式）
        q = torch.cat([q_pe, q_nope], dim=-1)  # 拼接回完整query，形状(batch,seq,n_heads,head_dim)
        k = self.wk(x)    # 计算共享key（所有头共用），形状(batch, seq, head_dim)
        k = self.k_norm(k)  # 对key做LayerNorm归一化，稳定训练
        k_pe, k_nope = torch.split(k, [self.rope_head_dim, self.head_dim - self.rope_head_dim], dim=-1)
        # 分离key的RoPE部分和非RoPE部分
        # 注意：unsqueeze(2)增加head维度以匹配apply_rotary_emb的4D输入格式
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis, False).squeeze(2)  # key的RoPE编码（非交错）
        k = torch.cat([k_pe, k_nope], dim=-1)  # 拼接回完整key，形状(batch,seq,head_dim)
        q = rotate_activation(q)  # 对query应用Hadamard变换（增强特征分散性）
        k = rotate_activation(k)  # 对key应用Hadamard变换
        q_fp8, q_scale = act_quant(q, block_size, self.scale_fmt)  # query量化为FP8
        k_fp8, k_scale = act_quant(k, block_size, self.scale_fmt)  # key量化为FP8
        self.k_cache[:bsz, start_pos:end_pos] = k_fp8  # 将当前步FP8 key存入缓存
        self.k_scale_cache[:bsz, start_pos:end_pos] = k_scale  # 将key缩放因子存入缓存
        weights = self.weights_proj(x.float()) * self.n_heads ** -0.5  # 计算头权重：(batch,seq,n_heads)，缩放1/sqrt(n_heads)
        weights = weights.unsqueeze(-1) * q_scale * self.softmax_scale  # 组合缩放因子：头权重×量化缩放×注意力缩放
        index_score = fp8_index(  # 调用FP8索引得分内核计算索引得分
            q_fp8.contiguous(),  # FP8量化的query
            weights,              # 组合缩放因子
            self.k_cache[:bsz, :end_pos].contiguous(),  # 历史FP8 key缓存
            self.k_scale_cache[:bsz, :end_pos].contiguous()  # 历史key缩放因子
        )  # 返回索引得分，形状(batch, seq, end_pos)
        if mask is not None:  # 如果有注意力掩码
            index_score += mask  # 加到得分上（-inf位置经softmax变为0）
        topk_indices = index_score.topk(min(self.index_topk, end_pos), dim=-1)[1]
        # 选出得分最高的top-k个key索引；[1]取索引值（[0]是得分值）
        topk_indices_ = topk_indices.clone()  # 克隆用于分布式验证
        dist.broadcast(topk_indices_, src=0)  # 从rank 0广播topk结果
        assert torch.all(topk_indices == topk_indices_), f"{topk_indices=} {topk_indices_=}"  # 验证所有进程结果一致
        return topk_indices  # 返回top-k位置索引


def weight_dequant(weight, scale):
    """
    将FP8量化权重反量化为默认浮点类型（通常BF16）。
    用于decode阶段预先反量化wkv_b权重，避免每步重复执行。

    Args:
        weight: FP8量化的2D权重矩阵，形状(out, in)
        scale: block-wise缩放因子，形状(out//128, in//128)
    Returns:
        反量化后的权重，形状与weight相同
    """
    shape = weight.shape  # 记录原始形状(out_features, in_features)
    assert weight.dim() == 2  # 断言为2D矩阵
    weight = weight.view(
        shape[0] // block_size, block_size, shape[1] // block_size, block_size
    ).transpose(1, 2).contiguous().view(-1, block_size * block_size)
    # 按block重组：将权重分成(out//128, 128, in//128, 128)块，转置后展平每个block
    weight = (weight.float() * scale.view(-1, 1).float()
              ).to(torch.get_default_dtype()
                   ).view(shape[0] // block_size, shape[1] // block_size, block_size, block_size
                          ).transpose(1, 2).contiguous().view(shape)
    # 每个block乘以对应缩放因子，转为默认dtype，再重组回原始形状
    return weight  # 返回反量化后的权重，形状(out, in)


class MLA(nn.Module):
    """
    Multi-Head Latent Attention (MLA) Layer.

    Attributes:
        dim (int): Dimensionality of the input features.
        n_heads (int): Number of attention heads.
        n_local_heads (int): Number of local attention heads for distributed systems.
        q_lora_rank (int): Rank for low-rank query projection.
        kv_lora_rank (int): Rank for low-rank key/value projection.
        qk_nope_head_dim (int): Dimensionality of non-positional query/key projections.
        qk_rope_head_dim (int): Dimensionality of rotary-positional query/key projections.
        qk_head_dim (int): Total dimensionality of query/key projections.
        v_head_dim (int): Dimensionality of value projections.
        softmax_scale (float): Scaling factor for softmax in attention computation.
    """
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.dim = args.dim
        self.n_heads = args.n_heads
        self.n_local_heads = args.n_heads // world_size
        self.q_lora_rank = args.q_lora_rank
        self.kv_lora_rank = args.kv_lora_rank
        self.qk_nope_head_dim = args.qk_nope_head_dim
        self.qk_rope_head_dim = args.qk_rope_head_dim
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim
        self.v_head_dim = args.v_head_dim

        self.wq_a = Linear(self.dim, self.q_lora_rank)  # Query第一阶段：dim → q_lora_rank（降维）
        self.q_norm = RMSNorm(self.q_lora_rank)  # 低秩空间的RMSNorm归一化
        self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)  # Query第二阶段：q_lora_rank → n_heads*qk_head_dim（列并行）
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)  # KV联合投影：dim → kv_lora_rank（潜在KV）+ rope_dim（位置编码）
        self.kv_norm = RMSNorm(self.kv_lora_rank)  # KV潜在表示的RMSNorm归一化
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))  # KV第二阶段：kv_lora_rank → n_heads*(qk_nope+v_dim)
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)  # 输出投影：n_heads*v_dim → dim（行并行）
        self.softmax_scale = self.qk_head_dim ** -0.5  # 标准注意力缩放因子 1/sqrt(qk_head_dim)
        self.scale_fmt = args.scale_fmt  # FP8缩放因子格式
        if args.max_seq_len > args.original_seq_len:  # 如果使用了YaRN长度外推
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0  # 计算YaRN注意力修正缩放
            self.softmax_scale = self.softmax_scale * mscale * mscale  # 应用mscale的平方（补偿长序列下注意力熵变化）

        self.indexer = Indexer(args)  # 创建稀疏注意力索引器

        self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)
        # KV低秩表示缓存（MLA核心优化）：只缓存压缩后的低秩表示，形状(max_batch, max_seq, kv_lora_rank)
        self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)
        # 位置编码缓存：单独缓存不可被低秩压缩的RoPE部分，形状(max_batch, max_seq, rope_dim)
        self.dequant_wkv_b = None  # decode阶段预反量化的wkv_b权重缓存（首次decode时初始化）

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        Forward pass for the Multi-Head Latent Attention (MLA) Layer.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, dim).
            start_pos (int): Starting position in the sequence for caching.
            freqs_cis (torch.Tensor): Precomputed complex exponential values for rotary embeddings.
            mask (Optional[torch.Tensor]): Mask tensor to exclude certain positions from attention.

        Returns:
            torch.Tensor: Output tensor with the same shape as the input.
        """
        bsz, seqlen, _ = x.size()  # 解包批量大小、序列长度、特征维度
        end_pos = start_pos + seqlen  # 当前序列在KV缓存中的结束位置
        qr = self.q_norm(self.wq_a(x))  # Query第一阶段：dim→q_lora_rank，并做RMSNorm，形状(batch,seq,q_lora_rank)
        q = self.wq_b(qr)  # Query第二阶段：q_lora_rank→n_local_heads*qk_head_dim
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)  # 重塑为多头形式
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)  # 分离非位置编码和位置编码部分
        q_pe = apply_rotary_emb(q_pe, freqs_cis)  # 对位置编码部分应用RoPE（交错模式）
        kv = self.wkv_a(x)  # KV联合投影：dim→kv_lora_rank+rope_dim
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)  # 分离潜在KV表示和key的RoPE部分
        kv = self.kv_norm(kv)  # 对KV潜在表示做RMSNorm归一化
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)  # 对key的RoPE部分应用位置编码（unsqueeze增加头维度）
        # 在实际部署中使用FP8 KV cache，这里通过量化-反量化模拟FP8精度损失
        kv_fp8, kv_scale = act_quant(kv, block_size, self.scale_fmt)  # KV低秩表示量化为FP8
        kv = (kv_fp8.view(-1, block_size).float() * kv_scale.view(-1, 1)).to(kv.dtype).view_as(kv)  # 立即反量化，模拟FP8精度
        self.kv_cache[:bsz, start_pos:end_pos] = kv  # 将低秩KV表示存入缓存
        self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)  # 将key的位置编码存入缓存
        if mask is not None:    # Prefill模式：处理整个序列（MHA）
            q = torch.cat([q_nope, q_pe], dim=-1)  # 拼接nope和pe，重组完整query
            kv = self.wkv_b(kv)  # 将低秩KV投影到完整KV空间
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)  # 重塑为多头形式
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)  # 分离key的nope部分和value
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)  # 拼接key的nope和pe部分
            scores = torch.einsum("bshd,bthd->bsht", q, k).mul_(self.softmax_scale)  # 计算注意力得分Q·K^T并缩放

            # 稀疏注意力：使用Indexer筛选top-k候选key
            topk_indices = self.indexer(x, qr, start_pos, freqs_cis, mask)  # 获取top-k索引
            index_mask = torch.full((bsz, seqlen, seqlen), float("-inf"), device=x.device).scatter_(-1, topk_indices, 0)  # 创建索引掩码：top-k位置为0，其余为-inf
            index_mask += mask  # 组合索引掩码与causal mask
            scores += index_mask.unsqueeze(2)  # 将组合掩码加到注意力得分

            scores = scores.softmax(dim=-1)  # softmax归一化得到注意力权重
            x = torch.einsum("bsht,bthd->bshd", scores, v)  # 用注意力权重聚合value
        else:                   # Decode模式：增量推理（MQA-like）
            if self.dequant_wkv_b is None and self.wkv_b.scale is not None:  # 首次decode且wkv_b是FP8时
                self.dequant_wkv_b = weight_dequant(self.wkv_b.weight, self.wkv_b.scale)  # 预反量化wkv_b权重
            wkv_b = self.wkv_b.weight if self.dequant_wkv_b is None else self.dequant_wkv_b  # 选择原始或预反量化权重
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)  # 重塑为多头格式
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            # 将q_nope投影到KV低秩空间（不显式构建K矩阵，直接在低秩空间计算）
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) +
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale
            # 分别计算nope部分（低秩空间）和pe部分的得分，求和并缩放

            # 稀疏注意力：使用Indexer筛选top-k候选key
            topk_indices = self.indexer(x, qr, start_pos, freqs_cis, mask)  # 获取top-k索引
            index_mask = torch.full((bsz, 1, end_pos), float("-inf"), device=x.device).scatter_(-1, topk_indices, 0)  # 创建索引掩码
            scores += index_mask.unsqueeze(2)  # 加到得分上

            scores = scores.softmax(dim=-1)  # softmax归一化
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])  # 在低秩空间聚合KV缓存
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])  # 投影到value空间
        x = self.wo(x.flatten(2))  # 合并所有头后通过输出投影，形状(batch, seq, dim)
        return x  # 返回注意力输出


class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used as a feed-forward layer.

    Attributes:
        w1 (nn.Module): Linear layer for input-to-hidden transformation.
        w2 (nn.Module): Linear layer for hidden-to-output transformation.
        w3 (nn.Module): Additional linear layer for feature transformation.
    """
    def __init__(self, dim: int, inter_dim: int, reduce_output: bool = True):
        """
        Initializes the MLP layer.

        Args:
            dim (int): Input and output dimensionality.
            inter_dim (int): Hidden layer dimensionality.
        """
        super().__init__()  # 调用父类初始化
        self.w1 = ColumnParallelLinear(dim, inter_dim)  # SwiGLU门控投影：dim→inter_dim（列并行）
        self.w2 = RowParallelLinear(inter_dim, dim, reduce_output=reduce_output)  # 降维投影：inter_dim→dim（行并行）
        self.w3 = ColumnParallelLinear(dim, inter_dim)  # SwiGLU值投影：dim→inter_dim（列并行）

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the MLP layer.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after MLP computation.
        """
        return self.w2((F.silu(self.w1(x).float()) * self.w3(x).float()).type_as(x))
        # SwiGLU：对w1输出做SiLU激活，逐元素乘以w3输出，转回输入dtype，再通过w2降维


class Gate(nn.Module):
    """
    Gating mechanism for routing inputs in a mixture-of-experts (MoE) model.

    Attributes:
        dim (int): Dimensionality of input features.
        topk (int): Number of top experts activated for each input.
        n_groups (int): Number of groups for routing.
        topk_groups (int): Number of groups to route inputs to.
        score_func (str): Scoring function ('softmax' or 'sigmoid').
        route_scale (float): Scaling factor for routing weights.
        weight (torch.nn.Parameter): Learnable weights for the gate.
        bias (Optional[torch.nn.Parameter]): Optional bias term for the gate.
    """
    def __init__(self, args: ModelArgs):
        """
        Initializes the Gate module.

        Args:
            args (ModelArgs): Model arguments containing gating parameters.
        """
        super().__init__()  # 调用父类初始化
        self.dim = args.dim  # 输入特征维度
        self.topk = args.n_activated_experts    # 每个token激活的路由专家数
        self.n_groups = args.n_expert_groups    # 专家分组数（1表示不分组）
        self.topk_groups = args.n_limited_groups  # 每次激活的最大专家组数
        self.score_func = args.score_func  # 路由得分函数：softmax或sigmoid
        self.route_scale = args.route_scale  # 路由权重缩放系数
        self.weight = nn.Parameter(torch.empty(args.n_routed_experts, args.dim))  # 门控权重矩阵，形状(n_experts, dim)
        self.bias = nn.Parameter(
            torch.empty(args.n_routed_experts, dtype=torch.float32)
        ) if self.dim == 7168 else None  # 仅在dim=7168（大模型）时使用偏置（负载均衡修正）

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass for the gating mechanism.

        Args:
            x (torch.Tensor): Input tensor, shape (num_tokens, dim)

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: Routing weights and selected expert indices.
        """
        scores = linear(x.float(), self.weight.float())  # 计算token与所有专家的相似度得分，形状(num_tokens, n_experts)，float32保精度
        if self.score_func == "softmax":  # softmax路由：归一化后的概率
            scores = scores.softmax(dim=-1)  # 对所有专家得分做softmax归一化
        else:  # sigmoid路由：每个专家独立打分
            scores = scores.sigmoid()  # 对每个专家得分独立做sigmoid
        original_scores = scores  # 保存原始得分（不受偏置影响），用于最终权重计算
        if self.bias is not None:  # 如果有负载均衡偏置
            scores = scores + self.bias  # 加上偏置修正（动态调整专家负载）
        if self.n_groups > 1:  # 使用专家分组路由策略
            scores = scores.view(x.size(0), self.n_groups, -1)  # 重塑为(num_tokens, n_groups, experts_per_group)
            if self.bias is None:  # 无偏置：用组内最大值作为组得分
                group_scores = scores.amax(dim=-1)  # 组级得分，形状(num_tokens, n_groups)
            else:  # 有偏置：用组内top-2得分之和作为组得分（DeepSeek策略）
                group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)  # 组内top-2求和，形状(num_tokens, n_groups)
            indices = group_scores.topk(self.topk_groups, dim=-1)[1]  # 选出得分最高的topk_groups个组索引
            mask = scores.new_ones(x.size(0), self.n_groups, dtype=bool).scatter_(1, indices, False)  # 创建组掩码：选中组为False，未选组为True
            scores = scores.masked_fill_(mask.unsqueeze(-1), float("-inf")).flatten(1)  # 将未选中组的专家得分设为-inf，再展平
        indices = scores.topk(self.topk, dim=-1)[1]  # 从候选专家中选出topk个，形状(num_tokens, topk)
        weights = original_scores.gather(1, indices)  # 从原始得分（无偏置）取出选中专家的权重，形状(num_tokens, topk)
        if self.score_func == "sigmoid":  # sigmoid路由需要归一化权重
            weights /= weights.sum(dim=-1, keepdim=True)  # 归一化使各专家权重和为1
        weights *= self.route_scale  # 乘以路由权重缩放系数
        return weights, indices  # 返回路由权重和专家索引


class Expert(nn.Module):
    """
    Expert layer for Mixture-of-Experts (MoE) models.

    Attributes:
        w1 (nn.Module): Linear layer for input-to-hidden transformation.
        w2 (nn.Module): Linear layer for hidden-to-output transformation.
        w3 (nn.Module): Additional linear layer for feature transformation.
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        Initializes the Expert layer.

        Args:
            dim (int): Input and output dimensionality.
            inter_dim (int): Hidden layer dimensionality.
        """
        super().__init__()  # 调用父类初始化
        self.w1 = Linear(dim, inter_dim)   # SwiGLU门控投影（普通Linear，非并行）
        self.w2 = Linear(inter_dim, dim)   # 降维投影
        self.w3 = Linear(dim, inter_dim)   # SwiGLU值投影

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the Expert layer (SwiGLU activation).

        Args:
            x (torch.Tensor): Input tensor, shape (num_routed_tokens, dim)

        Returns:
            torch.Tensor: Output tensor after expert computation.
        """
        return self.w2((F.silu(self.w1(x).float()) * self.w3(x).float()).type_as(x))
        # SwiGLU：SiLU(w1(x)) * w3(x)，再通过w2降维


class MoE(nn.Module):
    """
    Mixture-of-Experts (MoE) module.

    Attributes:
        dim (int): Dimensionality of input features.
        n_routed_experts (int): Total number of experts in the model.
        n_local_experts (int): Number of experts handled locally in distributed systems.
        n_activated_experts (int): Number of experts activated for each input.
        gate (nn.Module): Gating mechanism to route inputs to experts.
        experts (nn.ModuleList): List of expert modules.
        shared_experts (nn.Module): Shared experts applied to all inputs.
    """
    def __init__(self, args: ModelArgs):
        """
        Initializes the MoE module.

        Args:
            args (ModelArgs): Model arguments containing MoE parameters.
        """
        super().__init__()  # 调用父类初始化
        self.dim = args.dim  # 特征维度
        assert args.n_routed_experts % world_size == 0, \
            f"Number of experts must be divisible by world size (world_size={world_size})"  # 断言专家数能被进程数整除
        self.n_routed_experts = args.n_routed_experts  # 路由专家总数
        self.n_local_experts = args.n_routed_experts // world_size  # 本进程负责的本地专家数
        self.n_activated_experts = args.n_activated_experts  # 每token激活的路由专家数
        self.experts_start_idx = rank * self.n_local_experts  # 本进程专家起始全局编号
        self.experts_end_idx = self.experts_start_idx + self.n_local_experts  # 本进程专家结束全局编号（不含）
        self.gate = Gate(args)  # 创建路由门控模块
        self.experts = nn.ModuleList([
            Expert(args.dim, args.moe_inter_dim) if self.experts_start_idx <= i < self.experts_end_idx
            else None  # 本进程范围内创建Expert实例，其他设None节省内存
            for i in range(self.n_routed_experts)
        ])  # 创建全局专家列表，只有本进程范围的是实例
        self.shared_experts = MLP(
            args.dim, args.n_shared_experts * args.moe_inter_dim, reduce_output=False
        )  # 共享专家（所有token都经过），reduce_output=False延迟all_reduce

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the MoE module.

        Args:
            x (torch.Tensor): Input tensor, shape (batch, seq, dim)

        Returns:
            torch.Tensor: Output tensor after expert routing and computation.
        """
        shape = x.size()  # 保存原始形状(batch, seq, dim)
        x = x.view(-1, self.dim)  # 合并batch和seq维度，形状(-1, dim)
        weights, indices = self.gate(x)  # 计算路由权重和专家索引，各形状(num_tokens, topk)
        y = torch.zeros_like(x, dtype=torch.float32)  # 初始化输出累加器，float32保持精度
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts).tolist()  # 统计每个专家被路由到的token数
        for i in range(self.experts_start_idx, self.experts_end_idx):  # 只遍历本进程负责的专家
            if counts[i] == 0:  # 该专家没有被路由到token
                continue  # 跳过，避免空计算
            expert = self.experts[i]  # 获取第i个专家
            idx, top = torch.where(indices == i)  # 找出路由到专家i的token索引和topk位置
            y[idx] += expert(x[idx]) * weights[idx, top, None]  # 专家输出×路由权重，累加
        y += self.shared_experts(x)  # 加上共享专家的输出（所有token都经过）
        if world_size > 1:  # 多GPU模式
            dist.all_reduce(y)  # 对所有进程的专家输出求和
        return y.type_as(x).view(shape)  # 恢复输入dtype并reshape回原始形状


class Block(nn.Module):
    """
    Transformer block combining attention and feed-forward layers.

    Attributes:
        attn (nn.Module): Attention layer (MLA).
        ffn (nn.Module): Feed-forward network (MLP or MoE).
        attn_norm (nn.Module): Layer normalization for attention.
        ffn_norm (nn.Module): Layer normalization for feed-forward network.
    """
    def __init__(self, layer_id: int, args: ModelArgs):
        """
        Initializes the Transformer block.

        Args:
            layer_id (int): Layer index in the transformer.
            args (ModelArgs): Model arguments containing block parameters.
        """
        super().__init__()  # 调用父类初始化
        self.attn = MLA(args)  # 创建多头潜在注意力层（MLA）
        self.ffn = MLP(args.dim, args.inter_dim) if layer_id < args.n_dense_layers else MoE(args)
        # 前n_dense_layers层使用密集MLP，之后层使用MoE（稀疏激活）
        self.attn_norm = RMSNorm(args.dim)  # 注意力层前的Pre-Norm（RMSNorm）
        self.ffn_norm = RMSNorm(args.dim)   # FFN层前的Pre-Norm（RMSNorm）

    def forward(self, x: torch.Tensor, residual: torch.Tensor, start_pos: int,
                freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        """
        Forward pass for the Transformer block (Pre-Norm + 融合残差连接).

        Args:
            x (torch.Tensor): 上一步的归一化输出，形状(batch, seq, dim)
            residual (torch.Tensor): 残差累积张量；第一层时为None
            start_pos (int): KV缓存起始位置
            freqs_cis (torch.Tensor): 预计算的RoPE复数因子
            mask (Optional[torch.Tensor]): 注意力掩码

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: (当前层FFN输出, 更新后的残差)
        """
        if residual is None:  # 第一层：没有历史残差
            x, residual = self.attn_norm(x), x  # 先归一化，原始x作为残差起点
        else:  # 后续层：使用融合残差归一化
            x, residual = self.attn_norm(x, residual)  # 融合：x = normalize(x + residual)，同时更新residual
        x = self.attn(x, start_pos, freqs_cis, mask)  # 注意力层：输入归一化后的x
        x, residual = self.ffn_norm(x, residual)  # FFN前融合残差归一化
        x = self.ffn(x)  # FFN层（MLP或MoE）
        return x, residual  # 返回FFN输出和更新后的残差（传给下一层）


class Transformer(nn.Module):
    """
    Transformer model with positional embeddings, multiple layers, and output projection.

    Attributes:
        max_seq_len (int): Maximum sequence length for the transformer.
        embed (nn.Module): Embedding layer for input tokens.
        layers (torch.nn.ModuleList): List of transformer blocks.
        norm (nn.Module): Layer normalization applied after all blocks.
        head (nn.Module): Output projection layer mapping to vocabulary size.
        freqs_cis (torch.Tensor): Precomputed complex exponential values for rotary embeddings.
    """
    def __init__(self, args: ModelArgs):
        """
        Initializes the Transformer model.

        Args:
            args (ModelArgs): Model arguments containing transformer parameters.
        """
        global world_size, rank  # 声明修改全局变量
        world_size = dist.get_world_size() if dist.is_initialized() else 1  # 从分布式状态获取总进程数（未初始化则1）
        rank = dist.get_rank() if dist.is_initialized() else 0  # 从分布式状态获取当前进程编号（未初始化则0）
        Linear.dtype = torch.float8_e4m3fn if args.dtype == "fp8" else torch.bfloat16  # 根据配置设置全局权重类型
        Linear.scale_fmt = args.scale_fmt  # 设置全局FP8缩放因子格式
        super().__init__()  # 调用父类nn.Module初始化
        self.max_seq_len = args.max_seq_len  # 保存最大序列长度
        self.embed = ParallelEmbedding(args.vocab_size, args.dim)  # 创建支持张量并行的词嵌入层
        self.layers = torch.nn.ModuleList()  # 创建空的Transformer层列表
        for layer_id in range(args.n_layers):  # 循环创建所有Transformer层
            self.layers.append(Block(layer_id, args))  # 创建第layer_id层并添加（前几层MLP，后续MoE）
        self.norm = RMSNorm(args.dim)  # 最终的RMSNorm（在LM head之前）
        # lm_head存储为bf16，这里用fp32便于精确计算logits
        self.head = ColumnParallelLinear(args.dim, args.vocab_size, dtype=torch.float32)  # LM输出头：dim→vocab_size（列并行，float32）
        self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)
        # 预计算RoPE复数因子并注册为buffer，形状(max_seq_len, rope_dim//2)，不保存到checkpoint

    @torch.inference_mode()
    def forward(self, tokens: torch.Tensor, start_pos: int = 0):
        """
        Forward pass for the Transformer model.

        Args:
            tokens (torch.Tensor): Input tensor of token IDs with shape (batch_size, seq_len).
            start_pos (int, optional): Starting position in the sequence for rotary embeddings. Defaults to 0.

        Returns:
            torch.Tensor: Logits tensor of shape (batch_size, vocab_size).
        """
        seqlen = tokens.size(1)  # 获取当前输入序列长度
        freqs_cis = self.freqs_cis[start_pos:start_pos + seqlen]  # 取当前位置对应的RoPE因子，形状(seqlen, rope_dim//2)
        mask = torch.full(
            (seqlen, seqlen), float("-inf"), device=tokens.device
        ).triu_(1) if seqlen > 1 else None
        # Prefill时创建causal mask（上三角为-inf）；Decode时seq_len=1不需要mask
        h, residual = self.embed(tokens), None  # 词嵌入：(batch, seq)→(batch, seq, dim)；残差初始为None
        for layer in self.layers:  # 逐层通过Transformer Block
            h, residual = layer(h, residual, start_pos, freqs_cis, mask)  # 每层返回(x, residual)对
        h, _ = self.norm(h, residual)  # 最终归一化（融合最后一层残差），_丢弃更新后的residual
        logits = self.head(h[:, -1].float())  # 取最后位置特征，转float32，通过LM head计算logits
        if world_size > 1:  # 多GPU模式：聚合所有进程的词汇表分片
            all_logits = [torch.empty_like(logits) for _ in range(world_size)]  # 为每个进程创建接收缓冲区
            dist.all_gather(all_logits, logits)  # 收集所有进程的词汇表分片
            logits = torch.cat(all_logits, dim=-1)  # 拼接所有分片，恢复完整词汇表大小
        return logits  # 返回完整logits，形状(batch_size, vocab_size)


if __name__ == "__main__":
    torch.set_default_dtype(torch.bfloat16)     # 设置默认张量类型为BF16
    torch.set_default_device("cuda")             # 设置默认设备为CUDA
    torch.manual_seed(0)                         # 固定随机种子
    args = ModelArgs()                           # 使用默认参数创建ModelArgs实例
    x = torch.randint(0, args.vocab_size, (2, 128))  # 生成随机输入token（batch_size=2, seq_len=128）
    model = Transformer(args)                    # 创建完整Transformer模型
    print(model(x).size())                       # 打印输出logits形状（应为torch.Size([2, vocab_size])）